# 🎭 PoetryDuel-GPT — Colab Training (v3)

**Two-stage training with rhyme/tone/đối âm conditioning (~30M params).**  
**NEW: Curriculum Learning** — sort poems short→long, train progressively.
| Step | Time |
| Preprocess + tokenize | ~3 min |
| 🔍 Verify tokenizer | ~30 sec |
| Stage 1: All genres (10K steps) | ~3 hr T4 / ~2 hr L4 |
| 🔍 Verify Stage 1 | ~30 sec |
| Stage 2: Lục Bát fine-tune (5K steps) | ~1 hr T4 / ~40 min L4 |
| 🔍 Verify Stage 2 | ~30 sec |
| Generate poetry | ~1 min |
**To use Curriculum:** uncomment the `# CORPUS` and `# EXTRA` lines in Stage 1 cell.
Data is already in the repo (`poems_dataset.csv`). No Drive mount needed for data.
Mount Drive in last cell only to save checkpoints.


In [ ]:
# @title 1. Clone Repo + Install
!git clone https://github.com/weseegod/poetry-dual-gpt.git /content/poetry-dual-gpt 2>/dev/null
%cd /content/poetry-dual-gpt
!git pull origin main

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas tokenizers tqdm datasets
!mkdir -p checkpoints

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print('\n⚠️  Run cells in order. Each 🔍 cell verifies the previous step.')


In [ ]:
# @title 2. Download Data from Google Drive (~2 min)
# Upload data.zip to your Drive first (see README for how to generate)
from google.colab import drive
drive.mount('/content/drive')

# Copy data.zip from Drive (adjust path if needed)
!cp /content/drive/MyDrive/poetry-dual-gpt/data.zip /content/poetry-dual-gpt/ 2>/dev/null
!ls -lh /content/poetry-dual-gpt/data.zip

# Extract
!unzip -o data.zip -d data/ 2>/dev/null || echo 'data.zip not found in Drive. Upload it first.'
!ls data/
!wc -l data/poetry_corpus.txt data/doi_tho_corpus.txt data/poetry_corpus_combined.txt 2>/dev/null

# Verify tokenizer has correct special tokens
!python src/train_bpe.py --corpus data/poetry_corpus_combined.txt 2>/dev/null || echo 'Tokenizer exists, skipping'

print("\n✅ Data ready. Proceed to Stage 1.")


In [ ]:
# @title Verify: Tokenizer
%cd /content/poetry-dual-gpt
from tokenizers import Tokenizer
import os

tok_path = 'tokenizer/poetry_bpe.model'
if not os.path.exists(tok_path):
    print('Tokenizer not found. Run cell 2 first to download data + build tokenizer.')
else:
    tok = Tokenizer.from_file(tok_path)
    
    # Check special tokens are single IDs
    key_tokens = {
        '<|pad|>': 0, '<|start|>': 1, '<|reply|>': 2, '<|end|>': 3,
        '[LUC_BAT]': 4, '[DOI_THO]': 8, '<|linebreak|>': 9
    }
    all_ok = True
    for name, expected_id in key_tokens.items():
        actual = tok.token_to_id(name)
        ids = tok.encode(name).ids
        ok = actual == expected_id and len(ids) == 1
        if not ok: all_ok = False
        print(f'  {name:20s} id={actual} subwords={len(ids)} {"OK" if ok else "FAIL"}')
    
    print(f'\nVocab: {tok.get_vocab_size():,}')
    
    if all_ok:
        print('\nTokenizer OK - ready for Stage 1')
    else:
        print('\nTokenizer FAILED - do not proceed to training!')


In [ ]:
# @title 3. Stage 1 - Pretrain on ALL formats (~3 hr T4)
# Resume-safe: re-run if Colab disconnects
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> T4 GPU"

%cd /content/poetry-dual-gpt
import glob
# Combined corpus: 1.87M pairs (single-couplet + doi tho)
CORPUS = 'data/poetry_corpus_combined.txt'
EXTRA = ''

latest = sorted(glob.glob("checkpoints/stage1_step_*.pt"))
if latest:
    print(f"Resuming from {latest[-1]}")
    !python src/train.py --mode train --name stage1_ --corpus {CORPUS} {EXTRA} --resume {latest[-1]}
else:
    !python src/train.py --mode train --name stage1_ --corpus {CORPUS} {EXTRA}

print("\nStage 1 -> checkpoints/stage1_best.pt, stage1_final.pt")


In [ ]:
# @title Verify: Stage 1 (quick generation test)
%cd /content/poetry-dual-gpt
import os

ckpt = 'checkpoints/stage1_best.pt'
if not os.path.exists(ckpt):
    ckpt = 'checkpoints/stage1_final.pt'
if not os.path.exists(ckpt):
    print('No Stage 1 checkpoint found. Run Stage 1 training first.')
else:
    print(f'Testing: {ckpt}')
    !python src/sample.py --checkpoint {ckpt} --prompt "Than em nhu chen lua dong" --num_samples 2 --device cuda 2>&1 | head -30
    print('\nOutput should be 8-syllable Vietnamese verse. If garbled -> STOP.')


In [ ]:
# @title 5. Stage 2 - Fine-tune on Luc Bat only (~1 hr T4)
# Resumes from Stage 1 best checkpoint
assert torch.cuda.is_available(), "Enable GPU"

%cd /content/poetry-dual-gpt
import glob, os

# Prepare Luc Bat-only corpus from combined data
with open("data/poetry_corpus_combined.txt") as f:
    lines = f.readlines()
luc_bat = [l for l in lines if "[LUC_BAT]" in l or "[DOI_THO]" in l]
with open("data/corpus_luc_bat.txt", "w") as f:
    f.writelines(luc_bat)
lb_single = sum(1 for l in luc_bat if '[LUC_BAT]' in l)
lb_doi   = sum(1 for l in luc_bat if '[DOI_THO]' in l)
print(f'Luc Bat: {len(luc_bat):,} / {len(lines):,} total')
print(f'  [LUC_BAT]: {lb_single:,}  [DOI_THO]: {lb_doi:,}')

latest = sorted(glob.glob("checkpoints/stage2_step_*.pt"))
resume_from = latest[-1] if latest else "checkpoints/stage1_best.pt"
if not os.path.exists(resume_from) and not latest:
    resume_from = "checkpoints/stage1_final.pt"
print(f'Resuming from {resume_from}')

!python src/train.py \
    --name stage2_ \
    --resume {resume_from} \
    --corpus data/corpus_luc_bat.txt \
    --steps 5000

print("\nTwo-stage training complete!")


In [ ]:
# @title Verify: Stage 2 (compare both models)
%cd /content/poetry-dual-gpt
import os

for stage_name, ckpt_name in [('Stage 1', 'checkpoints/stage1_best.pt'), ('Stage 2', 'checkpoints/stage2_best.pt')]:
    if not os.path.exists(ckpt_name):
        alt = ckpt_name.replace('_best', '_final')
        if os.path.exists(alt):
            ckpt_name = alt
        else:
            print(f'{stage_name}: checkpoint not found, skipping')
            continue
    print(f'\n{"="*60}\n{stage_name}: {ckpt_name}\n{"="*60}')
    !python src/sample.py --checkpoint {ckpt_name} --prompt "Than em nhu chen lua dong" --num_samples 1 --device cuda 2>&1 | head -15

print('\nCompare: Stage 2 should be more fluid, better rhyme/tone than Stage 1.')


In [ ]:
# @title 6. Generate Poetry
%cd /content/poetry-dual-gpt

!python src/sample.py \
    --checkpoint checkpoints/stage2_best.pt \
    --temperature 0.75 \
    --top_p 0.92 \
    --num_samples 3

# Compare Stage 1 vs Stage 2:
# !python src/sample.py --checkpoint checkpoints/stage1_best.pt --num_samples 2

# Interactive (run in separate cell):
# !python src/sample.py --checkpoint checkpoints/stage2_best.pt --interactive

In [ ]:
# @title Save to Google Drive (tokenizer + checkpoints)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/poetry-dual-gpt"
!mkdir -p {DRIVE_DIR}/checkpoints

# Save tokenizer
!cp -v tokenizer/poetry_bpe.model {DRIVE_DIR}/
print('Tokenizer saved to Drive')

# Save all best checkpoints
import glob
for ckpt in sorted(glob.glob('checkpoints/stage*_best.pt') + glob.glob('checkpoints/stage*_final.pt')):
    !cp -v {ckpt} {DRIVE_DIR}/checkpoints/
print('\nAll saved to Drive -> poetry-dual-gpt/')
